# Análisis de Meteorología 2025

En este notebook se analiza información meteorológica de la Ciudad de México durante el año 2025.  
El objetivo principal es estudiar variables como humedad relativa y velocidad del viento para una estación específica.

## Carga y preparación de datos

El archivo original contiene algunas líneas iniciales con metadatos, por lo que la tabla real comienza después de las primeras 9 líneas.  
Por esta razón, al cargar el archivo con `pandas` se utiliza `skiprows=9`.

Las columnas principales del dataset son:

- `date`: fecha y hora de la medición
- `id_station`: identificador de la estación
- `id_parameter`: tipo de variable meteorológica medida
- `valor`: valor registrado
- `unit`: unidad de medición

In [13]:
import pandas as pd
import numpy as np


path = "/Users/luisenriquehernandeztorres/Desktop/Machine-Learning-Repo/meteorología_2025.csv"

df = pd.read_csv(path, skiprows =  9)

df.head(10)

,date,id_station,id_parameter,valor,unit
0,2025-01-01 00:00:00,ACO,TMP,8.7,5
1,2025-01-01 00:00:00,ACO,RH,28.0,6
2,2025-01-01 00:00:00,ACO,WSP,2.4,3
3,2025-01-01 00:00:00,ACO,WDR,2.0,4
4,2025-01-01 00:00:00,AJM,TMP,9.6,5
5,2025-01-01 00:00:00,AJM,RH,34.0,6
6,2025-01-01 00:00:00,AJM,WSP,4.2,3
7,2025-01-01 00:00:00,AJM,WDR,221.0,4
8,2025-01-01 00:00:00,AJU,TMP,1.5,5
9,2025-01-01 00:00:00,AJU,RH,74.0,6


In [14]:
sorted(df["id_station"].dropna().unique())

['ACO',
 'AJM',
 'AJU',
 'ATI',
 'BJU',
 'CAM',
 'CCA',
 'CHO',
 'CUA',
 'CUT',
 'FAC',
 'FAR',
 'GAM',
 'HGM',
 'INN',
 'IZT',
 'LAA',
 'LPR',
 'MER',
 'MGH',
 'MON',
 'MPA',
 'NEZ',
 'PED',
 'SAC',
 'SAG',
 'SFE',
 'TAH',
 'TLA',
 'TLI',
 'UAX',
 'UIZ',
 'VIF',
 'XAL']

## Selección de estación meteorológica

Para este análisis se trabaja únicamente con la estación `BJU`.  
Filtrar una sola estación permite estudiar el comportamiento meteorológico en un punto específico y evitar mezclar mediciones de diferentes zonas.

In [15]:
df_bju = df[df["id_station"] == "BJU"]

In [16]:
df_bju.head(10)

,date,id_station,id_parameter,valor,unit
16,2025-01-01 00:00:00,BJU,TMP,NaN,5
17,2025-01-01 00:00:00,BJU,RH,NaN,6
18,2025-01-01 00:00:00,BJU,WSP,NaN,3
19,2025-01-01 00:00:00,BJU,WDR,NaN,4
152,2025-01-01 01:00:00,BJU,TMP,NaN,5
153,2025-01-01 01:00:00,BJU,RH,NaN,6
154,2025-01-01 01:00:00,BJU,WSP,NaN,3
155,2025-01-01 01:00:00,BJU,WDR,NaN,4
288,2025-01-01 02:00:00,BJU,TMP,NaN,5
289,2025-01-01 02:00:00,BJU,RH,NaN,6


## Variables de interés

En este análisis se utilizan dos variables meteorológicas:

- `RH`: humedad relativa
- `WSP`: velocidad del viento

Se definen los siguientes eventos:

$$
H = \text{humedad alta} = RH \geq 70
$$

$$
V = \text{viento tranquilo} = WSP \leq 1
$$

Estos eventos permiten transformar las mediciones numéricas en condiciones booleanas: ocurre o no ocurre el evento.

## Probabilidad empírica

La probabilidad se calcula de forma empírica, es decir, usando la frecuencia observada en los datos.

La fórmula general es:

$$
P(A) = \frac{\text{número de veces que ocurre A}}{\text{número total de observaciones válidas}}
$$

Se eliminan los valores faltantes antes de calcular las probabilidades, ya que no representan mediciones reales.

In [17]:
df_bju = df[df["id_station"] == "BJU"].copy()

rh = df_bju[df_bju["id_parameter"] == "RH"]["valor"].dropna()
wsp = df_bju[df_bju["id_parameter"] == "WSP"]["valor"].dropna()

prob_humedad_alta = (rh >= 70).mean()
prob_viento_tranquilo = (wsp <= 1).mean()

print("Probabilidad de humedad alta:", prob_humedad_alta)
print("Probabilidad de viento tranquilo:", prob_viento_tranquilo)

Probabilidad de humedad alta: 0.23765272758561348
Probabilidad de viento tranquilo: 0.2551213634016182


## Probabilidad de viento tranquilo

La probabilidad de viento tranquilo se calcula como:

$$
P(V) = P(WSP \leq 1)
$$

En la estación `BJU`, esta probabilidad fue:

$$
P(WSP \leq 1) = 0.2551
$$

Esto significa que aproximadamente el **25.51%** de las mediciones válidas de velocidad del viento presentan viento tranquilo.

## Probabilidad condicional

La probabilidad condicional permite calcular la probabilidad de que ocurra un evento sabiendo que otro ya ocurrió.

La fórmula general es:

$$
P(A|B) = \frac{P(A \cap B)}{P(B)}
$$

En este caso se calculan dos probabilidades condicionales:

$$
P(H|V)
$$

y

$$
P(V|H)
$$

In [22]:
wide = df_bju.pivot_table(
    index="date",
    columns="id_parameter",
    values="valor",
    aggfunc="mean"
)

validos = wide[["RH", "WSP"]].dropna()

humedad_alta = validos["RH"] >= 70
viento_tranquilo = validos["WSP"] <= 1

p_humedad_dado_viento = (humedad_alta & viento_tranquilo).sum() / viento_tranquilo.sum()
p_viento_dado_humedad = (humedad_alta & viento_tranquilo).sum() / humedad_alta.sum()

print("Probabilidad de humedad alta dado viento tranquilo:", p_humedad_dado_viento)
print("Probabilidad de viento tranquilo dado humedad alta:", p_viento_dado_humedad)

Probabilidad de humedad alta dado viento tranquilo: 0.47031039136302294
Probabilidad de viento tranquilo dado humedad alta: 0.5047067342505431


## Humedad alta dado viento tranquilo

La probabilidad de humedad alta dado que hay viento tranquilo es:

$$
P(H|V) = P(RH \geq 70 | WSP \leq 1)
$$

Con los datos de la estación `BJU`:

$$
P(H|V) = \frac{697}{1482} = 0.4703
$$

Esto significa que cuando hay viento tranquilo, la probabilidad de observar humedad alta es de aproximadamente **47.03%**.

## Viento tranquilo dado humedad alta

La probabilidad de viento tranquilo dado que hay humedad alta es:

$$
P(V|H) = P(WSP \leq 1 | RH \geq 70)
$$

Con los datos de la estación `BJU`:

$$
P(V|H) = \frac{697}{1381} = 0.5047
$$

Esto significa que cuando hay humedad alta, la probabilidad de observar viento tranquilo es de aproximadamente **50.47%**.